###**Trabalho Final de Processamento de Imagens Biomédicas**

### 1 - Carregamento do Dataset

In [ ]:
import os
import glob
from google.colab import drive

!rm -rf /content/cbis_ddsm
print("Pasta antiga deletada com sucesso! O caminho está livre.")

# Monta o Drive
drive.mount('/content/drive')

# Definições
DRIVE_FOLDER = "/content/drive/MyDrive/datasets"
ZIP_FILE_NAME = "cbis-ddsm-breast-cancer-image-dataset.zip"
ZIP_PATH_DRIVE = os.path.join(DRIVE_FOLDER, ZIP_FILE_NAME)

# Garante que a pasta existe
os.makedirs(DRIVE_FOLDER, exist_ok=True)

# Lógica robusta de verificação
if os.path.exists(ZIP_PATH_DRIVE):
    print(f"O arquivo {ZIP_FILE_NAME} foi encontrado no Drive!")
else:
    print("O arquivo ZIP não existe no Drive. Iniciando download...")
    # O -p define o caminho de destino do download direto no Drive
    !kaggle datasets download -d awsaf49/cbis-ddsm-breast-cancer-image-dataset -p "{DRIVE_FOLDER}"

    # Verifica se o download realmente funcionou
    if os.path.exists(ZIP_PATH_DRIVE):
        print("Download bem-sucedido e salvo no Drive.")
    else:
        print("ERRO: O download não salvou o arquivo no Drive. Verifique seu espaço em disco.")

# Descompactação local (sempre no /content/)
DEST_DIR = "/content/cbis_ddsm"
if not os.path.exists(DEST_DIR):
    print("Extraindo arquivos para o Colab...")
    !unzip -q "{ZIP_PATH_DRIVE}" -d "{DEST_DIR}"
    print("Extração finalizada.")
else:
    print("Dataset já extraído no ambiente Colab.")

print("Total de imagens reais no disco:", len(glob.glob('/content/cbis_ddsm/**/*.jpg', recursive=True)))

### 2 - Instalação e Importação das Bibliotecas

In [ ]:
!pip uninstall -y sympy torch torchvision
!pip install sympy torch torchvision --upgrade
!pip install git+https://github.com/AIM-Harvard/pyradiomics

In [ ]:
import os
import re
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm import tqdm
import sympy
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset
import SimpleITK as sitk
from radiomics import featureextractor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configuração de sementes para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executando o pipeline em: {device}")

### 3 - Carregando CSV

In [ ]:
INPUT_DIR = 'cbis_ddsm/csv'

print("Carregando os arquivos de metadados CSV oficiais...")
mass_train = pd.read_csv(os.path.join(INPUT_DIR, 'mass_case_description_train_set.csv'))
mass_test = pd.read_csv(os.path.join(INPUT_DIR, 'mass_case_description_test_set.csv'))
dicom_info = pd.read_csv(os.path.join(INPUT_DIR, 'dicom_info.csv'))

# Filtrar Benignos e Malignos e padronizar o nome para 'Pathology'
mass_train = mass_train[mass_train['pathology'].isin(['BENIGN', 'MALIGNANT'])].reset_index(drop=True)
mass_test = mass_test[mass_test['pathology'].isin(['BENIGN', 'MALIGNANT'])].reset_index(drop=True)

mass_train = mass_train.rename(columns={'pathology': 'Pathology'})
mass_test = mass_test.rename(columns={'pathology': 'Pathology'})

# Extrair o UID da imagem recortada (Crop_UID)
def extract_uid(path):
    if pd.isna(path):
        return None
    parts = str(path).replace('\\', '/').split('/')
    return parts[-2] if len(parts) > 1 else None

mass_train['Crop_UID'] = mass_train['cropped image file path'].apply(extract_uid)
mass_test['Crop_UID'] = mass_test['cropped image file path'].apply(extract_uid)

# Filtrar o dicom_info para os recortes
# Ajustamos o nome da coluna para 'Cropped_Image_Path' para usar com a função do Otsu
roi_crops = dicom_info[dicom_info['SeriesDescription'] == 'cropped images'][['SeriesInstanceUID', 'image_path']].rename(
    columns={'SeriesInstanceUID': 'Crop_UID', 'image_path': 'Cropped_Image_Path'}
)

# Cruzar os dados para obter os caminhos finais das imagens cropped
df_train = pd.merge(mass_train, roi_crops, on='Crop_UID', how='inner')
df_test = pd.merge(mass_test, roi_crops, on='Crop_UID', how='inner')

print("\n" + "="*50)
print(f"Total de imagens cropped para TREINO: {len(df_train)}")
print(df_train['Pathology'].value_counts())
print("-" * 50)
print(f"Total de imagens cropped para TESTE: {len(df_test)}")
print(df_test['Pathology'].value_counts())
print("="*50)

In [ ]:
INPUT_DIR = 'cbis_ddsm/csv'

print("Carregando os arquivos de metadados CSV oficiais...")
mass_train = pd.read_csv(os.path.join(INPUT_DIR, 'mass_case_description_train_set.csv'))
mass_test = pd.read_csv(os.path.join(INPUT_DIR, 'mass_case_description_test_set.csv'))
dicom_info = pd.read_csv(os.path.join(INPUT_DIR, 'dicom_info.csv'))

mass_train = mass_train[mass_train['pathology'].isin(['BENIGN', 'MALIGNANT'])].reset_index(drop=True)
mass_test = mass_test[mass_test['pathology'].isin(['BENIGN', 'MALIGNANT'])].reset_index(drop=True)

mass_train = mass_train.rename(columns={'pathology': 'Pathology'})
mass_test = mass_test.rename(columns={'pathology': 'Pathology'})

def extract_uid(path):
    if pd.isna(path): return None
    parts = str(path).replace('\\', '/').split('/')
    return parts[-2] if len(parts) > 1 else None

# Extrai os 3 UIDs necessários!
for df in [mass_train, mass_test]:
    df['Crop_UID'] = df['cropped image file path'].apply(extract_uid)
    df['Mask_UID'] = df['ROI mask file path'].apply(extract_uid)
    df['Full_UID'] = df['image file path'].apply(extract_uid)

# Filtra o dicom_info para os 3 tipos de imagens
roi_crops = dicom_info[dicom_info['SeriesDescription'] == 'cropped images'][['SeriesInstanceUID', 'image_path']].rename(
    columns={'SeriesInstanceUID': 'Crop_UID', 'image_path': 'Cropped_Image_Path'}
)
roi_masks = dicom_info[dicom_info['SeriesDescription'] == 'ROI mask images'][['SeriesInstanceUID', 'image_path']].rename(
    columns={'SeriesInstanceUID': 'Mask_UID', 'image_path': 'Mask_Image_Path'}
)
full_imgs = dicom_info[dicom_info['SeriesDescription'] == 'full mammogram images'][['SeriesInstanceUID', 'image_path']].rename(
    columns={'SeriesInstanceUID': 'Full_UID', 'image_path': 'Full_Image_Path'}
)

# Cruzamentos
df_train = pd.merge(mass_train, roi_crops, on='Crop_UID', how='inner')
df_train = pd.merge(df_train, roi_masks, on='Mask_UID', how='inner')
df_train = pd.merge(df_train, full_imgs, on='Full_UID', how='inner')
df_test = pd.merge(mass_test, roi_crops, on='Crop_UID', how='inner')
df_test = pd.merge(df_test, roi_masks, on='Mask_UID', how='inner')
df_test = pd.merge(df_test, full_imgs, on='Full_UID', how='inner')

print(f"\nTREINO: {len(df_train)} imagens")
print(f"TESTE: {len(df_test)} imagens")

### 4 - Extração de Características (ResNet50)

In [ ]:
class CBISDDSMRapidoDataset(Dataset):
    def __init__(self, dataframe, transform=None, preprocessing_type=None):
        """
        preprocessing_type: Pode ser None, 'gaussian_blur', 'median_blur', ou 'clahe'
        """
        self.df = dataframe
        self.transform = transform
        self.preprocessing_type = preprocessing_type
        self.label_mapping = {'BENIGN': 0, 'MALIGNANT': 1}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rel_path = str(row['Cropped_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')

        base_path, _ = os.path.splitext(rel_path)
        full_path = rel_path
        for ext in ['.jpg', '.png', '.jpeg']:
            if os.path.exists(base_path + ext):
                full_path = base_path + ext
                break

        img = cv2.imread(full_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.ones((240, 240), dtype=np.uint8) * 128

        # Pré-processamento
        if self.preprocessing_type == 'gaussian_blur':
            img = cv2.GaussianBlur(img, (5, 5), 0)
        elif self.preprocessing_type == 'median_blur':
            img = cv2.medianBlur(img, 5)
        elif self.preprocessing_type == 'clahe':
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            img = clahe.apply(img)

        img_pil = Image.fromarray(img).convert('RGB')

        label = self.label_mapping[row['Pathology']]

        if self.transform:
            img_tensor = self.transform(img_pil)

        return img_tensor, label

transform_resnet = transforms.Compose([
    transforms.Resize((240, 240)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Inicializando a ResNet50 pré-treinada...")
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Identity()
model = model.to(device)
model.eval()

# Função para extração
def extrair_caracteristicas_resnet(df, modelo_resnet, transform, preproc_type, device):
    dataset = CBISDDSMRapidoDataset(df, transform=transform, preprocessing_type=preproc_type)
    loader = DataLoader(dataset, batch_size=16, shuffle=False)

    features_list = []
    labels_list = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc=f"ResNet50 ({preproc_type})"):
            inputs = inputs.to(device)
            outputs = modelo_resnet(inputs)
            features_list.append(outputs.cpu().numpy())
            labels_list.extend(labels.numpy())

    X_net = np.concatenate(features_list, axis=0)
    y_net = np.array(labels_list)
    return X_net, y_net

# Extração de características
FILTRO_RESNET = None

print("\nEXTRAÇÃO RESNET (TREINO)")
X_train_resnet, y_train = extrair_caracteristicas_resnet(df_train, model, transform_resnet, FILTRO_RESNET, device)

print("\nEXTRAÇÃO RESNET (TESTE)")
X_test_resnet, y_test = extrair_caracteristicas_resnet(df_test, model, transform_resnet, FILTRO_RESNET, device)

print(f"\nCaracterísticas da ResNet50 extraídas.")
print(f"Matriz de Treino: {X_train_resnet.shape} | Matriz de Teste: {X_test_resnet.shape}")


# Salvando no drive
print("\nSalvando características da ResNet50.")
df_export_train_resnet = pd.DataFrame(X_train_resnet)
df_export_train_resnet.insert(0, 'Pathology', y_train)
df_export_train_resnet.insert(0, 'ID_Imagem', df_train['Crop_UID'].values)
df_export_train_resnet.to_csv(os.path.join(DRIVE_FOLDER, 'features_resnet_train.csv'), index=False)
df_export_test_resnet = pd.DataFrame(X_test_resnet)
df_export_test_resnet.insert(0, 'Pathology', y_test)
df_export_test_resnet.insert(0, 'ID_Imagem', df_test['Crop_UID'].values)
df_export_test_resnet.to_csv(os.path.join(DRIVE_FOLDER, 'features_resnet_test.csv'), index=False)

print(f"Características da ResNet50 gerados e salvas no drive.")

### 5 - Extração de Características (Radiômica)


1. **Pré-processamento:** filtro bilateral (remove ruído preservando bordas) + CLAHE (realce de contraste local).
2. **Segmentação inicial:** limiarização de Otsu sobre a imagem realçada.
3. **Abertura morfológica:** rompe conexões finas entre o nódulo e o tecido denso vizinho.
4. **Seleção do componente central:** mantém apenas o blob mais próximo do centro do recorte (o crop do CBIS-DDSM já é centralizado na lesão), descartando ruído de borda.
5. **Fechamento morfológico:** garante uma máscara sólida.
6. **Contorno Ativo Morfológico (Chan-Vese):** refina a borda final ajustando-a à transição real de intensidade do nódulo.
7. **Fallback:** se tudo falhar, usa uma elipse central — nenhuma imagem é descartada do conjunto de dados.

In [ ]:
import os
import cv2
import logging
import numpy as np
import pandas as pd
import SimpleITK as sitk
import matplotlib.pyplot as plt
import random
from scipy import ndimage as ndi
from skimage.segmentation import morphological_chan_vese
from radiomics import featureextractor
logging.getLogger("radiomics").setLevel(logging.ERROR)

#=========================================================================================================================

def _selecionar_componente_central(mask_bin):
    """
    Isola o objeto mais central da máscara.
    """

    h, w = mask_bin.shape
    rotulos, n_componentes = ndi.label(mask_bin)
    if n_componentes == 0: return np.zeros_like(mask_bin)

    centro = np.array([h / 2.0, w / 2.0])
    melhor_label, melhor_score = None, np.inf

    # Avalia cada cluster. Ignora menores que 20 pixels.
    for comp_id in range(1, n_componentes + 1):
        ys, xs = np.where(rotulos == comp_id)
        tamanho = len(ys)
        if tamanho < 20: continue

        centroide = np.array([ys.mean(), xs.mean()])
        distancia = np.linalg.norm(centroide - centro)

        # O score penaliza clusters distantes e beneficia clusters maiores
        score = distancia / np.sqrt(tamanho)

        if score < melhor_score:
            melhor_score = score
            melhor_label = comp_id

    return (rotulos == melhor_label).astype(np.uint8) if melhor_label else np.zeros_like(mask_bin)

#=========================================================================================================================

def _refinar_com_chan_vese(img_contrastada, mask_inicial, max_dim=256, num_iter=15):
    """
    Usa contornos ativos (Chan-Vese) para abraçar melhor as bordas irregulares do nódulo.
    """

    h, w = img_contrastada.shape
    escala = min(1.0, max_dim / max(h, w))

    if escala < 1.0:
        novo_w, novo_h = max(1, int(w * escala)), max(1, int(h * escala))
        img_small = cv2.resize(img_contrastada, (novo_w, novo_h), interpolation=cv2.INTER_AREA)
        mask_small = cv2.resize(mask_inicial, (novo_w, novo_h), interpolation=cv2.INTER_NEAREST)
    else:
        img_small, mask_small = img_contrastada, mask_inicial

    img_norm = img_small.astype(float) / 255.0
    mask_cv = morphological_chan_vese(
        img_norm, num_iter=num_iter, init_level_set=mask_small.astype(np.uint8),
        smoothing=2, lambda1=1, lambda2=1
    ).astype(np.uint8)

    # Volta para a resolução original
    if escala < 1.0: mask_cv = cv2.resize(mask_cv, (w, h), interpolation=cv2.INTER_NEAREST)
    return mask_cv

#=========================================================================================================================

def segmentar_nodulo(img):
    """
    Aplica pré-processamento para destacar o nódulo.
    """

    h, w = img.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    area_minima = 0.01 * h * w

    # Filtro Bilateral e CLAHE
    img_denoised = cv2.bilateralFilter(img, d=5, sigmaColor=50, sigmaSpace=50)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_contrastada = clahe.apply(img_denoised)

    # Otsu
    img_blur = cv2.GaussianBlur(img_contrastada, (5, 5), 0)
    _, mask_otsu = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Abertura remove ruídos brancos, Fechamento fecha buracos nas bordas
    mask_aberta = cv2.morphologyEx(mask_otsu, cv2.MORPH_OPEN, kernel, iterations=2)
    mask_aberta = cv2.morphologyEx(mask_aberta, cv2.MORPH_CLOSE, kernel)

    # Isola o nódulo no centro
    mask_central = _selecionar_componente_central((mask_aberta > 0).astype(np.uint8))

    # Preenche buracos internos no nódulo encontrado
    if mask_central.sum() >= area_minima:
        mask_central = ndi.binary_fill_holes(mask_central).astype(np.uint8)
        mask_central = cv2.morphologyEx(mask_central * 255, cv2.MORPH_CLOSE, kernel)
        mask_central = (mask_central > 0).astype(np.uint8)

    mask_refinada = mask_central

    # Refinamento fino: Se a máscara inicial é viável, tenta abraçar a textura com Chan-Vese
    if mask_central.sum() >= area_minima:
        try:
            mask_cv = _refinar_com_chan_vese(img_contrastada, mask_central)

            # Limite de segurança: evita que o Chan-Vese "vaze" e engula a imagem inteira
            limite_seguranca = cv2.dilate(mask_central, kernel, iterations=3)
            mask_cv = mask_cv & limite_seguranca
            if mask_cv.sum() >= area_minima: mask_refinada = mask_cv
        except Exception: pass # Se der erro numérico no Chan-Vese, engole e segue com a máscara central

    usou_fallback = False

    # Se tudo falhar e o objeto sumir, cria uma elipse genérica no meio.
    if mask_refinada.sum() >= area_minima:
        mask_final = ndi.binary_fill_holes(mask_refinada).astype(np.uint8)
        mask_final = cv2.morphologyEx(mask_final * 255, cv2.MORPH_CLOSE, kernel)
        mask_final = (mask_final > 0).astype(np.uint8) * 255
    else:
        usou_fallback = True
        mask_final = np.zeros((h, w), dtype=np.uint8)
        cv2.ellipse(mask_final, (w // 2, h // 2), (int(w * 0.35), int(h * 0.35)), 0, 0, 360, 255, -1)

    return mask_final, usou_fallback

#=========================================================================================================================

def pipeline_radiomics_otsu(df_input, num_amostras_exibir=5):
    """
    Executa todo o processamento: carrega imagem, segmenta e extrai as caraterísticas da radiômica.
    """

    extractor = featureextractor.RadiomicsFeatureExtractor()
    extractor.disableAllFeatures()
    extractor.enableFeatureClassByName('shape2D')
    extractor.enableFeatureClassByName('firstorder')
    extractor.enableFeatureClassByName('glcm')
    extractor.enableFeatureClassByName('glrlm')
    extractor.enableFeatureClassByName('glszm')
    extractor.enableFeatureClassByName('gldm')
    extractor.enableFeatureClassByName('ngtdm')

    caracteristicas_lista, ids_processados, y_lista = [], [], []
    total = len(df_input)
    contador_exibidos = 0

    indices_para_exibir = random.sample(range(total), min(num_amostras_exibir, total))

    for i, (idx, row) in enumerate(df_input.iterrows(), 1):
        barra = '=' * int(i * 30 // total)
        print(f"\rProcessando: [{barra:<30}] {i}/{total} ({i*100//total}%)", end="", flush=True)

        rel_path = str(row['Cropped_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')
        base_crop, _ = os.path.splitext(rel_path)
        img_path = rel_path

        for ext in ['.jpg', '.png', '.jpeg']:
            if os.path.exists(base_crop + ext):
                img_path = base_crop + ext
                break

        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None: continue

        mask_calculada, usou_fallback = segmentar_nodulo(img)

        if idx in indices_para_exibir:
            mask_bool = mask_calculada.astype(bool)
            sobreposicao = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
            sobreposicao[mask_bool] = [255, 0, 0] # Destaca em vermelho
            fig, axes = plt.subplots(1, 3, figsize=(12, 4))
            axes[0].imshow(img, cmap='gray'); axes[0].set_title(f"Original: {row['Crop_UID']}")
            axes[1].imshow(mask_calculada, cmap='gray'); axes[1].set_title("Máscara Calculada")
            axes[2].imshow(sobreposicao); axes[2].set_title("Sobreposição")
            for ax in axes: ax.axis('off')
            plt.show()

        mask_final = (mask_calculada / 255).astype(np.uint8)
        if np.sum(mask_final) == 0: continue # Impede erro do SimpleITK se a máscara for nula

        # Converte pro formato que o Pyradiomics exige
        sitk_img = sitk.GetImageFromArray(img.astype(np.float32))
        sitk_mask = sitk.GetImageFromArray(mask_final)

        try:
            features = extractor.execute(sitk_img, sitk_mask)
            # Filtra os metadados que não são usadas
            f_numericas = {k: float(v) for k, v in features.items() if not k.startswith('diagnostics')}
            caracteristicas_lista.append(f_numericas)
            ids_processados.append(row['Crop_UID'])

            # Gera o alvo pro modelo binário final
            y_lista.append(1 if row['Pathology'] == 'MALIGNANT' else 0)
        except Exception: continue

    print()

    # Monta a matriz
    df_resultado = pd.DataFrame(caracteristicas_lista)
    df_resultado.insert(0, 'Pathology', y_lista)
    df_resultado.insert(0, 'ID_Imagem', ids_processados)
    return df_resultado

# Execução
print("\nEXTRAÇÃO TREINO")
df_train_radio = pipeline_radiomics_otsu(df_train)

print("\nEXTRAÇÃO TESTE")
df_test_radio = pipeline_radiomics_otsu(df_test)

print(f"Treino Médico: {df_train_radio.shape} | Teste Médico: {df_test_radio.shape}")

# Salvando no drive
df_train_radio.to_csv(os.path.join(DRIVE_FOLDER, 'features_radiomics_train_raw.csv'), index=False)
df_test_radio.to_csv(os.path.join(DRIVE_FOLDER, 'features_radiomics_test_raw.csv'), index=False)

### 5.1 - Extração de Características (Radiômica com as máscaras do **dataset**)


In [ ]:
import os
import cv2
import logging
import numpy as np
import pandas as pd
import SimpleITK as sitk
from radiomics import featureextractor
from tqdm.auto import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed
logging.getLogger("radiomics").setLevel(logging.ERROR)

#=========================================================================================================================

def get_existing_path(base_rel_path):
    """Encontra o arquivo real sem bater no disco cegamente repetidas vezes."""

    base_path, _ = os.path.splitext(base_rel_path)
    for ext in ['.jpg', '.png', '.jpeg']:
        temp_path = base_path + ext
        if os.path.exists(temp_path):
            return temp_path
    return None

#=========================================================================================================================

def process_single_row(row_dict):
    """Função para ser executada em paralelo por vários núcleos."""

    try:
        # Configura o extrator localmente para o worker
        extractor = featureextractor.RadiomicsFeatureExtractor()
        extractor.disableAllFeatures()
        extractor.enableFeatureClassByName('shape2D')
        extractor.enableFeatureClassByName('firstorder')
        for f_class in ['glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']:
            extractor.enableFeatureClassByName(f_class)
        extractor.settings['binWidth'] = 25
        extractor.settings['force2D'] = True

        # Configuração de caminhos
        img_rel = str(row_dict['Full_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')
        mask_rel = str(row_dict['Mask_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')

        img_path = get_existing_path(img_rel)
        mask_path = get_existing_path(mask_rel)

        if not img_path or not mask_path:
            return None

        # Leitura
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        img_mask_medico = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img is None or img_mask_medico is None:
            return None

        # Binarização
        _, mask_binaria = cv2.threshold(img_mask_medico, 127, 1, cv2.THRESH_BINARY)
        mask_final = mask_binaria.astype(np.uint8)

        if np.sum(mask_final) == 0:
            return None

        # Ignora os milhões de pixels pretos cortando a imagem em volta do tumor
        coords = cv2.findNonZero(mask_final)
        if coords is None:
            return None

        x, y, w, h = cv2.boundingRect(coords)
        pad = 20 # Margem de segurança de 20 pixels

        y1 = max(0, y - pad)
        y2 = min(img.shape[0], y + h + pad)
        x1 = max(0, x - pad)
        x2 = min(img.shape[1], x + w + pad)

        img_cropped = img[y1:y2, x1:x2]
        mask_cropped = mask_final[y1:y2, x1:x2]

        # Converte para SITK a matriz
        sitk_img = sitk.GetImageFromArray(img_cropped.astype(np.float32))
        sitk_mask = sitk.GetImageFromArray(mask_cropped)

        # Extração
        features = extractor.execute(sitk_img, sitk_mask)
        f_numericas = {k: float(v) for k, v in features.items() if not k.startswith('diagnostics')}

        f_numericas['ID_Imagem'] = row_dict['Crop_UID']
        f_numericas['Pathology'] = 1 if row_dict['Pathology'] == 'MALIGNANT' else 0

        return f_numericas

    except Exception as e:
        return None

#=========================================================================================================================

def pipeline_radiomics_medico(df_input):
    caracteristicas_lista = []

    # Converter DF para lista de dicionários
    rows_to_process = df_input.to_dict('records')
    total = len(rows_to_process)

    # Paralelismo
    max_workers = max(1, os.cpu_count() - 1)

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_single_row, row): row for row in rows_to_process}
        for future in tqdm(as_completed(futures), total=total, desc="Extraindo Padrão-Ouro (Paralelo)"):
            resultado = future.result()
            if resultado is not None:
                caracteristicas_lista.append(resultado)

    df_resultado = pd.DataFrame(caracteristicas_lista)

    if not df_resultado.empty:
        col_ordem = ['ID_Imagem', 'Pathology'] + [col for col in df_resultado.columns if col not in ['ID_Imagem', 'Pathology']]
        df_resultado = df_resultado[col_ordem]

    return df_resultado

# Execução
print("\nEXTRAÇÃO MÉDICA (TREINO)")
df_train_medico = pipeline_radiomics_medico(df_train)

print("\nEXTRAÇÃO MÉDICA (TESTE)")
df_test_medico = pipeline_radiomics_medico(df_test)

print(f"Treino Médico: {df_train_medico.shape} | Teste Médico: {df_test_medico.shape}")

# Salvando no drive
df_train_medico.to_csv(os.path.join(DRIVE_FOLDER, 'features_radiomics_train_medico_raw.csv'), index=False)
df_test_medico.to_csv(os.path.join(DRIVE_FOLDER, 'features_radiomics_test_medico_raw.csv'), index=False)

### 5.2 - Aplicando PCA nas radiômicas

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

#=========================================================================================================================

def processar_base_radiomica(train_csv, test_csv, n_components=0.95, random_state=42):
    """Carrega, trata NaNs, normaliza e retorna AMBOS: Sem PCA (std) e Com PCA."""
    df_train = pd.read_csv(os.path.join(DRIVE_FOLDER, train_csv))
    df_test = pd.read_csv(os.path.join(DRIVE_FOLDER, test_csv))

    # Isola metadados
    ids_train, ids_test = df_train['ID_Imagem'], df_test['ID_Imagem']
    y_train, y_test = df_train['Pathology'].values, df_test['Pathology'].values

    # Isola matriz numérica
    X_train_puro = df_train.drop(columns=['ID_Imagem', 'Pathology']).values
    X_test_puro = df_test.drop(columns=['ID_Imagem', 'Pathology']).values

    # Tratamento preventivo contra quebras na matemática (NaNs e Infinitos)
    X_train_puro = np.nan_to_num(X_train_puro, nan=0.0, posinf=0.0, neginf=0.0)
    X_test_puro = np.nan_to_num(X_test_puro, nan=0.0, posinf=0.0, neginf=0.0)

    # Normalização Z-score
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train_puro)
    X_test_std = scaler.transform(X_test_puro)

    # PCA
    pca = PCA(n_components=n_components, random_state=random_state)
    X_train_pca = pca.fit_transform(X_train_std)
    X_test_pca = pca.transform(X_test_std)

    return X_train_std, X_test_std, X_train_pca, X_test_pca, y_train, y_test, ids_train, ids_test

#=========================================================================================================================

def salvar_dataframe_final(X_matrix, y_values, ids_series, filename):
    """Reconstrói a estrutura padrão do dataset e exporta para o Drive."""

    df_export = pd.DataFrame(X_matrix)
    df_export.insert(0, 'Pathology', y_values)
    df_export.insert(0, 'ID_Imagem', ids_series.values)
    df_export.to_csv(os.path.join(DRIVE_FOLDER, filename), index=False)

# Carregando as radiômicas (Original/PCA)
print("Carregando e processando Radiômica (Otsu).")
X_train_otsu_std, X_test_otsu_std, X_train_otsu_pca, X_test_otsu_pca, y_train_otsu, y_test_otsu, ids_train_otsu, ids_test_otsu = \
    processar_base_radiomica('features_radiomics_train_raw.csv', 'features_radiomics_test_raw.csv')

print("Carregando e processando Radiômica Médica.")
X_train_medico_std, X_test_medico_std, X_train_medico_pca, X_test_medico_pca, y_train_medico, y_test_medico, ids_train_medico, ids_test_medico = \
    processar_base_radiomica('features_radiomics_train_medico_raw.csv', 'features_radiomics_test_medico_raw.csv')

# Carregando a ResNet50
print("\nCarregando os CSVs da ResNet50 do Drive...")
df_train_resnet = pd.read_csv(os.path.join(DRIVE_FOLDER, 'features_resnet_train.csv'))
df_test_resnet = pd.read_csv(os.path.join(DRIVE_FOLDER, 'features_resnet_test.csv'))
X_train_resnet_completo = df_train_resnet.drop(columns=['ID_Imagem', 'Pathology']).values
X_test_resnet_completo = df_test_resnet.drop(columns=['ID_Imagem', 'Pathology']).values

# Alinhamento e Combinação: ResNet50 + Radiômica Médica
print("\nAlinhando e combinando ResNet + Radiômica Médica.")
mascara_treino_medico = df_train_resnet['ID_Imagem'].isin(ids_train_medico)
mascara_teste_medico = df_test_resnet['ID_Imagem'].isin(ids_test_medico)
X_train_resnet_medico = X_train_resnet_completo[mascara_treino_medico]
X_test_resnet_medico = X_test_resnet_completo[mascara_teste_medico]

# Sem PCA
X_train_comb_medico_bruto = np.hstack((X_train_resnet_medico, X_train_medico_std))
X_test_comb_medico_bruto = np.hstack((X_test_resnet_medico, X_test_medico_std))

# Com PCA
X_train_comb_medico_pca = np.hstack((X_train_resnet_medico, X_train_medico_pca))
X_test_comb_medico_pca = np.hstack((X_test_resnet_medico, X_test_medico_pca))

# Alinhamento e Combinação: ResNet50 + Radiômica Otsu
print("Alinhando e combinando ResNet + Radiômica Otsu.")
mascara_treino_otsu = df_train_resnet['ID_Imagem'].isin(ids_train_otsu)
mascara_teste_otsu = df_test_resnet['ID_Imagem'].isin(ids_test_otsu)
X_train_resnet_otsu = X_train_resnet_completo[mascara_treino_otsu]
X_test_resnet_otsu = X_test_resnet_completo[mascara_teste_otsu]

# Sem PCA
X_train_comb_otsu_bruto = np.hstack((X_train_resnet_otsu, X_train_otsu_std))
X_test_comb_otsu_bruto = np.hstack((X_test_resnet_otsu, X_test_otsu_std))

# Com PCA
X_train_comb_otsu_pca = np.hstack((X_train_resnet_otsu, X_train_otsu_pca))
X_test_comb_otsu_bruto_pca = np.hstack((X_test_resnet_otsu, X_test_otsu_pca))


# Salando no drive
print("\nExportando os CSVs para o drive.")

# Radiômicas (PCA)
salvar_dataframe_final(X_train_otsu_pca, y_train_otsu, ids_train_otsu, 'features_radiomics_train_pca.csv')
salvar_dataframe_final(X_test_otsu_pca, y_test_otsu, ids_test_otsu, 'features_radiomics_test_pca.csv')
salvar_dataframe_final(X_train_medico_pca, y_train_medico, ids_train_medico, 'features_radiomics_medico_train_pca.csv')
salvar_dataframe_final(X_test_medico_pca, y_test_medico, ids_test_medico, 'features_radiomics_medico_test_pca.csv')

# Combinações da Radiômica Médica
salvar_dataframe_final(X_train_comb_medico_bruto, y_train_medico, ids_train_medico, 'features_combined_medico_train_bruto.csv')
salvar_dataframe_final(X_test_comb_medico_bruto, y_test_medico, ids_test_medico, 'features_combined_medico_test_bruto.csv')
salvar_dataframe_final(X_train_comb_medico_pca, y_train_medico, ids_train_medico, 'features_combined_medico_train_pca.csv')
salvar_dataframe_final(X_test_comb_medico_pca, y_test_medico, ids_test_medico, 'features_combined_medico_test_pca.csv')

# Combinações da Radiômica Otsu
salvar_dataframe_final(X_train_comb_otsu_bruto, y_train_otsu, ids_train_otsu, 'features_combined_train_bruto.csv')
salvar_dataframe_final(X_test_comb_otsu_bruto, y_test_otsu, ids_test_otsu, 'features_combined_test_bruto.csv')
salvar_dataframe_final(X_train_comb_otsu_pca, y_train_otsu, ids_train_otsu, 'features_combined_train_pca.csv')
salvar_dataframe_final(X_test_comb_otsu_bruto_pca, y_test_otsu, ids_test_otsu, 'features_combined_test_pca.csv')

print("Todos os CSVs (Originais e PCA) salvos no drive.")

### 6 - Funções de Avaliação

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
LABELS = ['Benigno (0)', 'Maligno (1)']

#=========================================================================================================================

def calcular_metricas(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    return {
        'Acurácia': accuracy_score(y_true, y_pred),
        'Sensibilidade': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
        'Especificidade': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1-Score': f1_score(y_true, y_pred, pos_label=1),
        'Matriz_Confusao': cm
    }

#=========================================================================================================================

def treinar_avaliar(modelo, nome_modelo, nome_conjunto, X_train, X_test, y_train, y_test, usar_pca=False):
    scaler = StandardScaler()
    X_tr_final = scaler.fit_transform(X_train)
    X_te_final = scaler.transform(X_test)

    # Se PCA necessário
    if usar_pca:
        pca = PCA(n_components=0.95, random_state=42)
        X_tr_final = pca.fit_transform(X_tr_final)
        X_te_final = pca.transform(X_te_final)

    modelo.fit(X_tr_final, y_train)
    y_pred = modelo.predict(X_te_final)

    metricas = calcular_metricas(y_test, y_pred)
    print(f"[{nome_modelo}] {nome_conjunto} -> Acc: {metricas['Acurácia']*100:.1f}% | F1: {metricas['F1-Score']:.3f}")

    return {
        'Modelo': nome_modelo,
        'Características': nome_conjunto,
        'Acurácia': metricas['Acurácia'],
        'Sensibilidade': metricas['Sensibilidade'],
        'Especificidade': metricas['Especificidade'],
        'F1-Score': metricas['F1-Score'],
        'Matriz': metricas['Matriz_Confusao']
    }

In [ ]:
# ALINHAMENTO: Garante que as features da ResNet correspondam
# exatamente aos IDs que sobreviveram na extração radiômica

# Mapeia Crop_UID → posição no array da ResNet (que veio do df_train/df_test completo)
uid_para_idx_train = {uid: i for i, uid in enumerate(df_train['Crop_UID'].values)}
uid_para_idx_test  = {uid: i for i, uid in enumerate(df_test['Crop_UID'].values)}

# IDs que a radiômica conseguiu processar
ids_radio_train = df_train_radio['ID_Imagem'].values
ids_radio_test  = df_test_radio['ID_Imagem'].values

# Seleciona apenas as linhas da ResNet cujo UID está na radiômica, na mesma ordem
X_train_resnet_alinhado = np.array([X_train_resnet[uid_para_idx_train[uid]] for uid in ids_radio_train])
X_test_resnet_alinhado  = np.array([X_test_resnet[uid_para_idx_test[uid]]  for uid in ids_radio_test])

print(f"ResNet alinhada → Treino: {X_train_resnet_alinhado.shape} | Teste: {X_test_resnet_alinhado.shape}")
print(f"Radiômica Otsu  → Treino: {X_train_otsu.shape} | Teste: {X_test_otsu.shape}")

In [ ]:
import pandas as pd

# Radiômica Otsu
X_train_otsu = df_train_radio.drop(columns=['Pathology', 'ID_Imagem']).reset_index(drop=True)
X_test_otsu  = df_test_radio.drop(columns=['Pathology', 'ID_Imagem']).reset_index(drop=True)
y_train_otsu = df_train_radio['Pathology'].reset_index(drop=True)
y_test_otsu  = df_test_radio['Pathology'].reset_index(drop=True)

# Radiômica Médico
X_train_medico = df_train_medico.drop(columns=['Pathology', 'ID_Imagem']).reset_index(drop=True)
X_test_medico  = df_test_medico.drop(columns=['Pathology', 'ID_Imagem']).reset_index(drop=True)
y_train_medico = df_train_medico['Pathology'].reset_index(drop=True)
y_test_medico  = df_test_medico['Pathology'].reset_index(drop=True)

# ResNet Pura
X_train_resnet = pd.DataFrame(X_train_resnet_alinhado).reset_index(drop=True)
X_test_resnet  = pd.DataFrame(X_test_resnet_alinhado).reset_index(drop=True)

# Colunas como string para evitar bugs no Sklearn
X_train_otsu_resnet = pd.concat([X_train_resnet, X_train_otsu], axis=1)
X_test_otsu_resnet  = pd.concat([X_test_resnet, X_test_otsu], axis=1)
X_train_otsu_resnet.columns = [str(c) for c in X_train_otsu_resnet.columns]
X_test_otsu_resnet.columns  = [str(c) for c in X_test_otsu_resnet.columns]

X_train_resnet_medico = pd.concat([X_train_resnet, X_train_medico], axis=1)
X_test_resnet_medico  = pd.concat([X_test_resnet, X_test_medico], axis=1)
X_train_resnet_medico.columns = [str(c) for c in X_train_resnet_medico.columns]
X_test_resnet_medico.columns  = [str(c) for c in X_test_resnet_medico.columns]


# Antes e Depois do PCA para tudo que tem Radiômica
conjuntos_features = {
    #   Puras
    '1. Radiômica Médico - Bruto': (X_train_medico, X_test_medico, y_train_medico, y_test_medico, False),
    '2. Radiômica Médico - c/ PCA': (X_train_medico, X_test_medico, y_train_medico, y_test_medico, True),

    '3. Radiômica Otsu - Bruto': (X_train_otsu, X_test_otsu, y_train_otsu, y_test_otsu, False),
    '4. Radiômica Otsu - c/ PCA': (X_train_otsu, X_test_otsu, y_train_otsu, y_test_otsu, True),

    # Fusões com Otsu
    '5. Radiômica Otsu + ResNet - Bruto': (X_train_otsu_resnet, X_test_otsu_resnet, y_train_otsu, y_test_otsu, False),
    '6. Radiômica Otsu + ResNet - c/ PCA': (X_train_otsu_resnet, X_test_otsu_resnet, y_train_otsu, y_test_otsu, True),

    # Fusões com Médico
    '7. ResNet + Radiômica Médico - Bruto': (X_train_resnet_medico, X_test_resnet_medico, y_train_medico, y_test_medico, False),
    '8. ResNet + Radiômica Médico - c/ PCA': (X_train_resnet_medico, X_test_resnet_medico, y_train_medico, y_test_medico, True),

    # ResNet Pura
    '9. ResNet Pura': (X_train_resnet, X_test_resnet, y_train_medico, y_test_medico, False)
}

# Limpa a lista antes de treinar
resultados = []

print(f"{len(conjuntos_features)} abordagens prontas para comparação direta.\n")
for nome, (X_tr, _, _, _, pca_flag) in conjuntos_features.items():
    status_pca = "Ativado" if pca_flag else "Desativado"
    print(f"-> {nome:<40} | Colunas Iniciais: {X_tr.shape[1]:<5} | PCA: {status_pca}")

### 7 - Classificadores (SVM & Random Forest)

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

# Modelos -> Parâmetros
modelos_para_testar = {
    'SVM (RBF)': SVC(kernel='rbf', C=2.0, gamma='scale', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42)
}

resultados = []

print("INICIANDO CLASSIFICAÇÃO DOS MODELOS")

# Executa os dois modelos de uma vez
for nome_conjunto, (X_tr, X_te, y_tr, y_te, usa_pca) in conjuntos_features.items():
    print(f"\n>>> Processando características: {nome_conjunto}")

    for nome_modelo, modelo_instancia in modelos_para_testar.items():
        resultado_metricas = treinar_avaliar(
            modelo=modelo_instancia,
            nome_modelo=nome_modelo,
            nome_conjunto=nome_conjunto,
            X_train=X_tr,
            X_test=X_te,
            y_train=y_tr,
            y_test=y_te,
            usar_pca=usa_pca
        )

        # Salva os resultados
        resultados.append(resultado_metricas)

print("\nTreinamento concluído.")

### 8 - Resultados Obtidos

In [ ]:
import os
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Tabela Comparativa Geral
df_resultados = pd.DataFrame(resultados)
df_resultados_fmt = df_resultados.copy()
for col in ['Acurácia', 'Sensibilidade', 'Especificidade', 'F1-Score']:
    df_resultados_fmt[col] = (df_resultados_fmt[col] * 100).round(2)

# Ajusta a tabela geral
tabela_exibicao = df_resultados_fmt.drop(columns=['Matriz']).sort_values(by=['Modelo', 'Características']).reset_index(drop=True)

print("================ TABELA COMPARATIVA GERAL ================")
display(tabela_exibicao)
tabela_exibicao.to_csv(os.path.join(DRIVE_FOLDER, 'resultados_finais_completos.csv'), index=False)

melhor_f1 = df_resultados.loc[df_resultados['F1-Score'].idxmax()]
print(f"\nMELHOR COMBINAÇÃO (F1-Score): {melhor_f1['Modelo']} + {melhor_f1['Características']} -> {melhor_f1['F1-Score']:.3f}")

# Gráficos
metricas_plot = ['Acurácia', 'Sensibilidade', 'Especificidade', 'F1-Score']
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

for ax, metrica in zip(axes, metricas_plot):
    pivot = df_resultados.pivot(index='Características', columns='Modelo', values=metrica)
    pivot.plot(kind='bar', ax=ax, legend=(metrica == 'Acurácia'), color=['#1f77b4', '#ff7f0e'])
    ax.set_title(metrica, fontweight='bold')
    ax.set_ylim(0, 1)
    ax.set_xticklabels(pivot.index, rotation=30, ha='right')
    ax.set_ylabel(metrica)
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

# Matrizes de Confusão
num_testes = len(resultados)
colunas = 6
linhas = math.ceil(num_testes / colunas)
fig, axes = plt.subplots(linhas, colunas, figsize=(24, 4 * linhas))
axes = axes.flatten()

for i, row in enumerate(resultados):
    sns.heatmap(row['Matriz'], annot=True, fmt='d', cmap='Blues',
                xticklabels=LABELS, yticklabels=LABELS, cbar=False, ax=axes[i])
    axes[i].set_xlabel('Predito')
    axes[i].set_ylabel('Real')

    titulo = f"{row['Modelo']}\n{row['Características']}"
    axes[i].set_title(titulo, fontsize=10, pad=10)

for j in range(num_testes, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

### 8.1 - Análise da segmentação por Dice e IoU

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from tqdm.auto import tqdm

#=========================================================================================================================

def calcular_dice_iou(mask_pred, mask_gt):
    """Calcula o coeficiente Dice e a métrica IoU entre duas máscaras."""

    m_p = (mask_pred > 0).astype(bool)
    m_g = (mask_gt > 0).astype(bool)

    intersection = np.logical_and(m_p, m_g).sum()
    union = np.logical_or(m_p, m_g).sum()

    iou = intersection / union if union > 0 else 0.0
    dice = 2 * intersection / (m_p.sum() + m_g.sum()) if (m_p.sum() + m_g.sum()) > 0 else 0.0

    return dice, iou

#=========================================================================================================================

print("Iniciando a avaliação da segmentação usando DICE e IoU.")
N_AVALIAR = min(100, len(df_test))
df_amostra = df_test.sample(N_AVALIAR, random_state=42)
resultados_segmentacao = []

for idx, row in tqdm(df_amostra.iterrows(), total=N_AVALIAR, desc="Calculando Métricas"):
    crop_path = str(row['Cropped_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')
    base_crop, _ = os.path.splitext(crop_path)
    img_crop_path = crop_path
    for ext in ['.jpg', '.png', '.jpeg']:
        if os.path.exists(base_crop + ext):
            img_crop_path = base_crop + ext; break

    full_path = str(row['Full_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')
    base_full, _ = os.path.splitext(full_path)
    img_full_path = full_path
    for ext in ['.jpg', '.png', '.jpeg']:
        if os.path.exists(base_full + ext):
            img_full_path = base_full + ext; break

    mask_path = str(row['Mask_Image_Path']).replace('CBIS-DDSM/', 'cbis_ddsm/')
    base_mask, _ = os.path.splitext(mask_path)
    img_mask_path = mask_path
    for ext in ['.jpg', '.png', '.jpeg']:
        if os.path.exists(base_mask + ext):
            img_mask_path = base_mask + ext; break

    if not os.path.exists(img_crop_path) or not os.path.exists(img_mask_path):
        continue

    img_crop = cv2.imread(img_crop_path, cv2.IMREAD_GRAYSCALE)
    mask_gt_full = cv2.imread(img_mask_path, cv2.IMREAD_GRAYSCALE)

    if img_crop is None or mask_gt_full is None:
        continue

    # Utiliza a função criada na radiômica de otsu
    mask_otsu, _ = segmentar_nodulo(img_crop)

    # Alinha as Máscaras
    if img_crop.shape == mask_gt_full.shape:
        mask_gt_crop = mask_gt_full
    else:
        if os.path.exists(img_full_path):
            img_full = cv2.imread(img_full_path, cv2.IMREAD_GRAYSCALE)
            # Usa a técnica de Template Matching
            if img_full is not None:
                # Procura a imagem pequena dentro da grande
                res = cv2.matchTemplate(img_full, img_crop, cv2.TM_CCOEFF_NORMED)
                _, _, _, max_loc = cv2.minMaxLoc(res)
                h, w = img_crop.shape
                # Recorta a máscara na mesma coordenada
                mask_gt_crop = mask_gt_full[max_loc[1]:max_loc[1]+h, max_loc[0]:max_loc[0]+w]
            else:
                continue
        else:
            continue

    # Binariza a máscara médica e valida os tamanhos
    _, mask_gt_bin = cv2.threshold(mask_gt_crop, 127, 1, cv2.THRESH_BINARY)
    mask_gt_crop = (mask_gt_bin * 255).astype(np.uint8)

    if mask_otsu.shape != mask_gt_crop.shape:
        continue

    # Calcula as métricas
    dice, iou = calcular_dice_iou(mask_otsu, mask_gt_crop)

    resultados_segmentacao.append({
        'UID': row['Crop_UID'],
        'Dice': dice,
        'IoU': iou,
        'img_crop': img_crop,
        'mask_otsu': mask_otsu,
        'mask_gt': mask_gt_crop
    })

In [ ]:
# Exibição dos Resultados
df_metricas = pd.DataFrame(resultados_segmentacao)

print(f"Métricas Médias da Segmentação (Amostra: {len(df_metricas)} imagens)")
print(f"{'='*60}")
print(f"🔹 Índice Dice Médio: {df_metricas['Dice'].mean():.4f}")
print(f"🔹 Índice IoU Médio:  {df_metricas['IoU'].mean():.4f}")
print(f"{'='*60}")

# Pega os 3 melhores e os 3 piores
df_metricas = df_metricas.sort_values(by='Dice', ascending=False).reset_index(drop=True)
melhores = df_metricas.head(3)
piores = df_metricas[df_metricas['Dice'] > 0.05].tail(3)

#=========================================================================================================================

def plotar_sobreposicao(lista_exemplos, titulo_geral):
    fig, axes = plt.subplots(len(lista_exemplos), 4, figsize=(16, 4 * len(lista_exemplos)))
    fig.suptitle(titulo_geral, fontsize=16, fontweight='bold', y=1.02)

    for i, (_, row) in enumerate(lista_exemplos.iterrows()):
        img = row['img_crop']
        m_otsu = row['mask_otsu']
        m_gt = row['mask_gt']

        sobreposicao = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        sobreposicao[m_otsu > 0] = [255, 0, 0]
        sobreposicao[m_gt > 0] = [0, 255, 0]
        sobreposicao[(m_otsu > 0) & (m_gt > 0)] = [255, 255, 0]

        ax_orig = axes[i, 0] if len(lista_exemplos) > 1 else axes[0]
        ax_otsu = axes[i, 1] if len(lista_exemplos) > 1 else axes[1]
        ax_gt   = axes[i, 2] if len(lista_exemplos) > 1 else axes[2]
        ax_over = axes[i, 3] if len(lista_exemplos) > 1 else axes[3]

        ax_orig.imshow(img, cmap='gray'); ax_orig.set_title(f"Imagem Original\nUID: {str(row['UID'])[-8:]}")
        ax_otsu.imshow(m_otsu, cmap='gray'); ax_otsu.set_title("Máscara Segmentação")
        ax_gt.imshow(m_gt, cmap='gray'); ax_gt.set_title("Máscara Médica")
        ax_over.imshow(sobreposicao); ax_over.set_title(f"Dice: {row['Dice']:.3f} | IoU: {row['IoU']:.3f}")

        for ax in [ax_orig, ax_otsu, ax_gt, ax_over]: ax.axis('off')

    plt.tight_layout()
    plt.show()

print("\nLegenda:")
print("Amarelo: Interseção (Verdadeiro Positivo)")
print("Vermelho: Algoritmo vazou para tecido saudável (Falso Positivo)")
print("Verde: Algoritmo não pegou uma parte do tumor (Falso Negativo)\n")

plotar_sobreposicao(melhores, "Melhores Resultados (3)")
plotar_sobreposicao(piores, "Piores Resultados (3)")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

pasta_salvamento = "/content/drive/MyDrive/datasets/"
df_resultados = pd.DataFrame(resultados)

# ── Gráfico de Barras ───────────────────────────────────────────────
print("[*] Gerando Gráfico de Barras Comparativo...")

plt.figure(figsize=(14, 6))
sns.barplot(data=df_resultados, x='Características', y='F1-Score',
            hue='Modelo', palette='viridis')
plt.title('Comparativo de F1-Score por Abordagem', fontsize=14)
plt.ylabel('F1-Score', fontsize=12)
plt.xlabel('Conjunto de Features', fontsize=12)
plt.ylim(0, 1.0)
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.legend(title='Classificador')
plt.tight_layout()

caminho_barras = os.path.join(pasta_salvamento, 'grafico_barras_resultados.png')
plt.savefig(caminho_barras, dpi=300)
plt.show()
print(f"[+] Gráfico salvo em: {caminho_barras}\n")


# ── Matrizes de Confusão (top 2) ────────────────────────────────────
print("[*] Gerando Painel de Matrizes de Confusão...")

# Pega direto do resultados —
def pegar_resultado(modelo, caracteristicas):
    for r in resultados:
        if r['Modelo'] == modelo and r['Características'] == caracteristicas:
            return r
    return None

campeao   = pegar_resultado('SVM (RBF)', '5. Radiômica Otsu + ResNet - Bruto')
vice      = pegar_resultado('SVM (RBF)', '9. ResNet Pura')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, res, cmap, titulo in zip(
    axes,
    [campeao, vice],
    ['Blues', 'Oranges'],
    [
        f"Melhor Geral: SVM (RBF)\nOtsu + ResNet — F1: {campeao['F1-Score']*100:.2f}%",
        f"Melhor ResNet Pura: SVM (RBF)\nResNet Pura — F1: {vice['F1-Score']*100:.2f}%"
    ]
):
    sns.heatmap(res['Matriz'], annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Benigno', 'Maligno'],
                yticklabels=['Benigno', 'Maligno'],
                cbar=False)
    ax.set_title(titulo, fontsize=11)
    ax.set_ylabel('Rótulo Real')
    ax.set_xlabel('Rótulo Predito')

plt.tight_layout()
caminho_matrizes = os.path.join(pasta_salvamento, 'painel_matrizes_confusao.png')
plt.savefig(caminho_matrizes, dpi=300)
plt.show()
print(f"[+] Painel de Matrizes salvo em: {caminho_matrizes}")